In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import make_scorer, accuracy_score, precision_score, recall_score, f1_score, cohen_kappa_score

In [3]:
dt = pd.read_excel('data.xlsx')

In [4]:
dt.head()

,Marital status,Application mode,Application order,Course,Daytime/evening attendance\t,Previous qualification,Previous qualification (grade),Nacionality,Mother's qualification,Father's qualification,...,Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
0,1,17,5,171,1,1,122.0,1,19,12,...,0,0,0,0,0.0,0,2025-08-10,2025-04-01 00:00:00,1974-01-01 00:00:00,Dropout
1,1,15,1,9254,1,1,160.0,1,1,3,...,0,6,6,6,13666666666666600,0,2025-09-13,-0.3,0.79,Graduate
2,1,1,5,9070,1,1,122.0,1,37,37,...,0,6,0,0,0.0,0,2025-08-10,2025-04-01 00:00:00,1974-01-01 00:00:00,Dropout
3,1,17,2,9773,1,1,122.0,1,38,37,...,0,6,10,5,2025-04-12 00:00:00,0,2025-04-09,-0.8,-3.12,Graduate
4,2,39,1,8014,0,1,100.0,1,37,38,...,0,6,6,6,13.0,0,2025-09-13,-0.3,0.79,Graduate


In [ ]:
dt.columns

Index(['Marital status', 'Application mode', 'Application order', 'Course',
       'Daytime/evening attendance\t', 'Previous qualification',
       'Previous qualification (grade)', 'Nacionality',
       'Mother's qualification', 'Father's qualification',
       'Mother's occupation', 'Father's occupation', 'Admission grade',
       'Displaced', 'Educational special needs', 'Debtor',
       'Tuition fees up to date', 'Gender', 'Scholarship holder',
       'Age at enrollment', 'International',
       'Curricular units 1st sem (credited)',
       'Curricular units 1st sem (enrolled)',
       'Curricular units 1st sem (evaluations)',
       'Curricular units 1st sem (approved)',
       'Curricular units 1st sem (grade)',
       'Curricular units 1st sem (without evaluations)',
       'Curricular units 2nd sem (credited)',
       'Curricular units 2nd sem (enrolled)',
       'Curricular units 2nd sem (evaluations)',
       'Curricular units 2nd sem (approved)',
       'Curricular units 2nd

In [5]:
dt.isna().sum()

,0
Marital status,0
Application mode,0
Application order,0
Course,0
Daytime/evening attendance\t,0
Previous qualification,0
Previous qualification (grade),0
Nacionality,0
Mother's qualification,0
Father's qualification,0


In [6]:
dt = dt.drop(["Curricular units 1st sem (grade)", "Curricular units 2nd sem (grade)", "Inflation rate"], axis=1)

In [7]:
dt.shape

(4424, 34)

In [ ]:
# Target sütununu sayısala çevirmek: Dropout=0, Enrolled=1, Graduate=2

# Label Encoding yaparak öğrenmeyi kolaylaştırıyoruz
dt["Target"] = dt["Target"].map({
    "Dropout": 0,
    "Enrolled": 1,
    "Graduate": 2
})

In [9]:
print(dt["Target"].value_counts())  # Kaç tane 0, 1, 2 olduğunu görmek için

Target
2    2209
0    1421
1     794
Name: count, dtype: int64


In [ ]:
print(dt["Target"].dtype)  

int64


In [11]:
X = dt.drop('Target', axis=1)
y = dt['Target']

In [ ]:
print(np.unique(y))  
print(y.dtype)       

[0 1 2]
int64


In [13]:
X = X.select_dtypes(include=[np.number])

In [14]:
# Değerlendirme metrikleri
scoring = {
    'accuracy': 'accuracy',
    'precision': make_scorer(precision_score, average='weighted', zero_division=0),
    'recall': make_scorer(recall_score, average='weighted'),
    'f1': make_scorer(f1_score, average='weighted'),
    'kappa': make_scorer(cohen_kappa_score)
}

In [ ]:
from sklearn.model_selection import KFold, cross_validate
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, cohen_kappa_score, make_scorer

# K-fold değerleri ve sonuç listeleri
k_values = [5, 10]
results_naive = []


for k in k_values:
    kf = KFold(n_splits=k, shuffle=True, random_state=42)

    model = GaussianNB()

    cv_results = cross_validate(model, X, y, cv=kf, scoring=scoring, n_jobs=-1)

    # Sonuçları sakla
    results_naive.append({
        "K": k,
        "Accuracy": np.mean(cv_results['test_accuracy']),
        "Precision": np.mean(cv_results['test_precision']),
        "Recall": np.mean(cv_results['test_recall']),
        "F1-score": np.mean(cv_results['test_f1']),
        "Cohen’s Kappa": np.mean(cv_results['test_kappa'])
    })

# Sonuçları tablo olarak gösterelim
results_naive = pd.DataFrame(results_naive)
print(results_naive)

    K  Accuracy  Precision    Recall  F1-score  Cohen’s Kappa
0   5  0.666821   0.649319  0.666821  0.655103       0.442265
1  10  0.670214   0.651939  0.670214  0.657993       0.447329


In [ ]:
from sklearn.model_selection import KFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

X = dt.drop('Target', axis=1)
y = dt['Target']

X = X.select_dtypes(include=[np.number])

# Veri ölçeklendirme 
scaler = StandardScaler()
X = scaler.fit_transform(X)

# K-fold değerleri ve sonuç listeleri
k_values = [5, 10]
results_knn = []

# K-NN modelini oluştur
knn = KNeighborsClassifier(n_neighbors=5, n_jobs=-1)  # n_neighbors'ı değiştirebiliriz

# Her K için çapraz doğrulama uygula
for k in k_values:
    kf = KFold(n_splits=k, shuffle=True, random_state=42)

    cv_results = cross_validate(knn, X, y, cv=kf, scoring=scoring, n_jobs=-1)

    # Sonuçları sakla
    results_knn.append({
        "K": k,
        "Accuracy": np.mean(cv_results['test_accuracy']),
        "Precision": np.mean(cv_results['test_precision']),
        "Recall": np.mean(cv_results['test_recall']),
        "F1-score": np.mean(cv_results['test_f1']),
        "Cohen’s Kappa": np.mean(cv_results['test_kappa'])
    })


results_knn = pd.DataFrame(results_knn)
print(results_knn)

    K  Accuracy  Precision    Recall  F1-score  Cohen’s Kappa
0   5  0.701628   0.682012  0.701628  0.687660       0.498689
1  10  0.698462   0.681963  0.698462  0.685867       0.493987


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, cross_validate
from sklearn.tree import DecisionTreeClassifier


k_values = [5, 10]
results_dt = []

# Decision Tree modelini oluşturma
tree = DecisionTreeClassifier(criterion='gini', max_depth=None, min_samples_split=2, min_samples_leaf=1)


for k in k_values:
    kf = KFold(n_splits=k, shuffle=True, random_state=42)

    cv_results = cross_validate(tree, X, y, cv=kf, scoring=scoring, n_jobs=-1)

    # Sonuçları sakla
    results_dt.append({
        "K": k,
        "Accuracy": np.mean(cv_results['test_accuracy']),
        "Precision": np.mean(cv_results['test_precision']),
        "Recall": np.mean(cv_results['test_recall']),
        "F1-score": np.mean(cv_results['test_f1']),
        "Cohen’s Kappa": np.mean(cv_results['test_kappa'])
    })

# Sonuçları tablo olarak gösterelim
results_dt = pd.DataFrame(results_dt)
print(results_dt)

    K  Accuracy  Precision    Recall  F1-score  Cohen’s Kappa
0   5  0.671343   0.678049  0.671343  0.674098       0.469674
1  10  0.673828   0.680042  0.673828  0.675874       0.472847


In [ ]:
from sklearn.model_selection import KFold, cross_validate
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

# K-fold değerleri ve sonuç listeleri
k_values = [5, 10]
results_svm = []

# SVM modelini oluşturma 
svm = SVC(kernel='rbf', C=1.0, gamma='scale')

# Her K için çapraz doğrulama uygula
for k in k_values:
    kf = KFold(n_splits=k, shuffle=True, random_state=42)

    cv_results = cross_validate(svm, X, y, cv=kf, scoring=scoring, n_jobs=-1)

    # Sonuçları sakla
    results_svm.append({
        "K": k,
        "Accuracy": np.mean(cv_results['test_accuracy']),
        "Precision": np.mean(cv_results['test_precision']),
        "Recall": np.mean(cv_results['test_recall']),
        "F1-score": np.mean(cv_results['test_f1']),
        "Cohen’s Kappa": np.mean(cv_results['test_kappa'])
    })

# Sonuçları tablo olarak gösterelim
results_svm = pd.DataFrame(results_svm)
print(results_svm)

    K  Accuracy  Precision    Recall  F1-score  Cohen’s Kappa
0   5  0.760850   0.743633  0.760850  0.740836       0.590534
1  10  0.757015   0.739880  0.757015  0.737524       0.584638


In [19]:
from sklearn.model_selection import KFold, cross_validate
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

# K-fold değerleri ve sonuç listeleri
k_values = [5, 10]
results_mlp = []

# MLP modelini oluştur
mlp = MLPClassifier(hidden_layer_sizes=(100,), activation='relu', solver='adam', max_iter=500)

# Her K için çapraz doğrulama uygula
for k in k_values:
    kf = KFold(n_splits=k, shuffle=True, random_state=42)

    cv_results = cross_validate(mlp, X, y, cv=kf, scoring=scoring, n_jobs=-1)

    # Sonuçları sakla
    results_mlp.append({
        "K": k,
        "Accuracy": np.mean(cv_results['test_accuracy']),
        "Precision": np.mean(cv_results['test_precision']),
        "Recall": np.mean(cv_results['test_recall']),
        "F1-score": np.mean(cv_results['test_f1']),
        "Cohen’s Kappa": np.mean(cv_results['test_kappa'])
    })

# Sonuçları tablo olarak gösterelim
results_mlp = pd.DataFrame(results_mlp)
print(results_mlp)

    K  Accuracy  Precision    Recall  F1-score  Cohen’s Kappa
0   5  0.724687   0.721343  0.724687  0.721962       0.548350
1  10  0.725609   0.717830  0.725609  0.719681       0.547061


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, cross_validate
from sklearn.ensemble import RandomForestClassifier

# K-fold değerleri ve sonuç listeleri
k_values = [5, 10]
results_rf = []

# Random Forest modelini oluşturma
rf = RandomForestClassifier(n_estimators=100, criterion='gini', max_depth=None, min_samples_split=2, bootstrap=True, n_jobs=-1)

# Her K için çapraz doğrulama uygulama
for k in k_values:
    kf = KFold(n_splits=k, shuffle=True, random_state=42)

    cv_results = cross_validate(rf, X, y, cv=kf, scoring=scoring, n_jobs=-1)

    # Sonuçları sakla
    results_rf.append({
        "K": k,
        "Accuracy": np.mean(cv_results['test_accuracy']),
        "Precision": np.mean(cv_results['test_precision']),
        "Recall": np.mean(cv_results['test_recall']),
        "F1-score": np.mean(cv_results['test_f1']),
        "Cohen’s Kappa": np.mean(cv_results['test_kappa'])
    })

# Sonuçları tablo olarak gösterelim
results_rf = pd.DataFrame(results_rf)
print(results_rf)

    K  Accuracy  Precision    Recall  F1-score  Cohen’s Kappa
0   5  0.778031   0.766343  0.778031  0.762746       0.623174
1  10  0.773291   0.759977  0.773291  0.757623       0.616154


In [ ]:
from sklearn.model_selection import KFold, cross_validate
from xgboost import XGBClassifier

scaler = StandardScaler()
X = scaler.fit_transform(X)

# K-fold değerleri ve sonuç listeleri
k_values = [5, 10]
results_xgb = []

# XGBoost modelini oluşturma
xgb = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, subsample=0.8, colsample_bytree=0.8, n_jobs=-1)

# Her K için çapraz doğrulama uygulama
for k in k_values:
    kf = KFold(n_splits=k, shuffle=True, random_state=42)

    cv_results = cross_validate(xgb, X, y, cv=kf, scoring=scoring, n_jobs=-1)

    # Sonuçları sakla
    results_xgb.append({
        "K": k,
        "Accuracy": np.mean(cv_results['test_accuracy']),
        "Precision": np.mean(cv_results['test_precision']),
        "Recall": np.mean(cv_results['test_recall']),
        "F1-score": np.mean(cv_results['test_f1']),
        "Cohen’s Kappa": np.mean(cv_results['test_kappa'])
    })

# Sonuçları tablo olarak gösterelim
results_xgb = pd.DataFrame(results_xgb)
print(results_xgb)

    K  Accuracy  Precision    Recall  F1-score  Cohen’s Kappa
0   5  0.775317   0.763177  0.775317  0.762555       0.620160
1  10  0.774880   0.765304  0.774880  0.762511       0.619601


In [ ]:
# Her sonuç setine model ismi ekleme
df_knn = pd.DataFrame(results_knn)
df_knn["Model"] = "K-NN"

df_dt = pd.DataFrame(results_dt)
df_dt["Model"] = "Decision Tree"

df_mlp = pd.DataFrame(results_mlp)
df_mlp["Model"] = "MLP"

df_naive = pd.DataFrame(results_naive)
df_naive["Model"] = "Naive Bayes"

df_rf = pd.DataFrame(results_rf)
df_rf["Model"] = "Random Forest"

df_svm = pd.DataFrame(results_svm)
df_svm["Model"] = "SVM"

df_xgb = pd.DataFrame(results_xgb)
df_xgb["Model"] = "XGB"


# Tüm DataFrame'leri birleştirerek tabloyu oluştur
df_all_results = pd.concat([df_knn, df_dt, df_mlp, df_naive, df_rf, df_svm, df_xgb], ignore_index=True)

# Sütun sırasını düzenleyelim
df_all_results = df_all_results[["Model", "K", "Accuracy", "Precision", "Recall", "F1-score", "Cohen’s Kappa"]]

# Tabloyu ekrana yazdır
print(df_all_results.to_string(index=False))

        Model  K  Accuracy  Precision   Recall  F1-score  Cohen’s Kappa
         K-NN  5  0.701628   0.682012 0.701628  0.687660       0.498689
         K-NN 10  0.698462   0.681963 0.698462  0.685867       0.493987
Decision Tree  5  0.671343   0.678049 0.671343  0.674098       0.469674
Decision Tree 10  0.673828   0.680042 0.673828  0.675874       0.472847
          MLP  5  0.724687   0.721343 0.724687  0.721962       0.548350
          MLP 10  0.725609   0.717830 0.725609  0.719681       0.547061
  Naive Bayes  5  0.666821   0.649319 0.666821  0.655103       0.442265
  Naive Bayes 10  0.670214   0.651939 0.670214  0.657993       0.447329
Random Forest  5  0.778031   0.766343 0.778031  0.762746       0.623174
Random Forest 10  0.773291   0.759977 0.773291  0.757623       0.616154
          SVM  5  0.760850   0.743633 0.760850  0.740836       0.590534
          SVM 10  0.757015   0.739880 0.757015  0.737524       0.584638
          XGB  5  0.775317   0.763177 0.775317  0.762555       0